# PFE Navigator — Système RAG

## 1. Présentation du projet

PFE Navigator est un système intelligent basé sur
la génération augmentée par récupération (RAG).

Il permet aux étudiants de poser des questions concernant
leur Projet de Fin d'Études (PFE) et d'obtenir des réponses
à partir d'un ensemble de documents PDF.

## 2. Objectifs

- Exploiter un corpus de cinq documents PDF.
- Tester un LLM seul, sans contexte documentaire.
- Construire un système de recherche documentaire.
- Générer des réponses fondées sur les passages récupérés.
- Afficher les sources utilisées.
- Comparer les résultats du LLM seul et du RAG.
- Identifier et corriger un échec du système.

 ## Étape 1 — Importer les bibliothèques

In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai

print("Bibliothèques importées avec succès.")

Bibliothèques importées avec succès.


### 1.1  Configuration Gemini

In [6]:
# Charger les variables du fichier .env
load_dotenv()

# Récupérer la clé API
api_key = os.getenv("GEMINI_API_KEY")

# Vérifier la clé API
if not api_key:
    raise ValueError("Clé API introuvable dans le fichier .env")

# Initialiser le client Gemini
client = genai.Client(api_key=api_key)

# Définir le modèle
MODEL = "gemini-3.8-flash"

print("Configuration terminée.")
print("Modèle utilisé :", MODEL)

Configuration terminée.
Modèle utilisé : gemini-3.8-flash


### 1.2  Définir les questions de test

In [7]:
questions = [
    "Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?",

    "Comment est calculée la note finale du PFE, et à partir de quelle note est-il validé ?",

    "Quelle police et quel interligne faut-il utiliser dans le rapport ?",

    "Quelle est la date de la session de rattrapage des soutenances ?",

    "Quel est le montant de la bourse accordée aux étudiants effectuant leur PFE ?"
]

print("Nombre de questions :", len(questions))

Nombre de questions : 5


## étape 2 - Tester le LLM seul

In [10]:

import time
import json
import os

resultats_llm = []
os.makedirs("../results", exist_ok=True)

for i, question in enumerate(questions, start=1):

    print(f"\n{'='*60}")
    print(f"QUESTION {i} : {question}")
    print("="*60)

    reponse = None
    erreur = None

    for tentative in range(1, 4):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=question
            )

            reponse = response.text or "Aucune réponse retournée."
            print("\nRéponse du LLM :")
            print(reponse)
            break

        except Exception as e:
            erreur = str(e)
            print(f"\nTentative {tentative} échouée : {erreur}")

            if tentative < 3:
                attente = tentative * 10
                print(f"Nouvelle tentative dans {attente} secondes...")
                time.sleep(attente)

    resultat = {
        "question": question,
        "reponse": reponse,
        "statut": "success" if reponse else "error",
        "erreur": erreur
    }

    resultats_llm.append(resultat)

    with open("../results/llm_sans_rag.json", "w", encoding="utf-8") as fichier:
        json.dump(resultats_llm, fichier, ensure_ascii=False, indent=4)

print("\nTest terminé.")



QUESTION 1 : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

Réponse du LLM :
Cette information dépend entièrement de **votre établissement** (université, école d'ingénieurs, institut), de votre filière et de votre calendrier académique. 

Pour avoir la réponse exacte, vous devez consulter les documents internes de votre école. Cependant, voici **comment cela se passe généralement** et **où trouver l'information** :

---

### 1. En règle générale :

* **La date limite :**
  * **Session normale (printemps/été) :** Souvent située entre **fin mai et début juillet** (environ 1 à 2 semaines avant la date de votre soutenance).
  * **Session de rattrapage ou stages longs :** Souvent entre **fin août et mi-septembre**.

* **La forme de remise :**
  * **Version numérique (obligatoire presque partout) :** 
    * Fichier au format **PDF** unique (souvent nommé selon une convention précise, ex. : *PFE_2024_NOM_Prenom.pdf*).
    * Dépôt sur l

In [11]:
os.makedirs("../results", exist_ok=True)

with open(
    "../results/llm_sans_rag.json",
    "w",
    encoding="utf-8"
) as fichier:

    json.dump(
        resultats_llm,
        fichier,
        ensure_ascii=False,
        indent=4
    )

print("Résultats sauvegardés.")

Résultats sauvegardés.


## étape 3 - Préparer les documents

### 3.1 Charger et extraire le contenu des PDF

Dans cette étape, nous chargeons les cinq documents PDF de notre corpus. Nous extrayons le texte page par page afin de conserver les informations nécessaires à la citation des sources. Ensuite, nous nettoyons le texte pour supprimer les espaces inutiles et les retours à la ligne superflus.

In [12]:
import re
from pathlib import Path
from pypdf import PdfReader

# Localiser le dossier contenant les PDF
dossier_documents = Path("../documents")

if not dossier_documents.exists():
    dossier_documents = Path("documents")

if not dossier_documents.exists():
    raise FileNotFoundError("Le dossier documents est introuvable.")

fichiers_pdf = sorted(dossier_documents.glob("*.pdf"))

if not fichiers_pdf:
    raise FileNotFoundError("Aucun fichier PDF trouvé dans le dossier documents.")

# Extraire le texte page par page
pages_data = []

for fichier in fichiers_pdf:
    lecteur = PdfReader(str(fichier))

    for numero_page, page in enumerate(lecteur.pages, start=1):
        texte = page.extract_text() or ""

        # Nettoyage du texte
        texte = re.sub(r"\s+", " ", texte)
        texte = texte.strip()

        if texte:
            pages_data.append({
                "source": fichier.name,
                "page": numero_page,
                "texte": texte
            })

print("Nombre de fichiers PDF :", len(fichiers_pdf))
print("Nombre de pages contenant du texte :", len(pages_data))

print("\nDocuments chargés :")
for fichier in fichiers_pdf:
    print("-", fichier.name)

Nombre de fichiers PDF : 5
Nombre de pages contenant du texte : 13

Documents chargés :
- 01_guide_pfe.pdf
- 02_reglement_stages.pdf
- 03_guide_redaction.pdf
- 04_guide_soutenance.pdf
- 05_calendrier_universitaire.pdf


### 3.2 - Découper les documents en chunks

Après l'extraction et le nettoyage, nous découpons les textes en morceaux de 400 mots avec un chevauchement de 50 mots. Ce chevauchement permet de conserver le contexte entre deux morceaux consécutifs. Chaque morceau garde le nom du document et les numéros des pages correspondantes.

In [13]:
taille_chunk = 400
overlap = 50
pas = taille_chunk - overlap

chunks = []

# Regrouper les pages par document
documents = {}

for page in pages_data:
    documents.setdefault(page["source"], []).append(page)

# Découpage document par document
for nom_document, pages in documents.items():

    mots_document = []
    pages_mots = []

    for page in pages:
        mots_page = page["texte"].split()

        mots_document.extend(mots_page)
        pages_mots.extend([page["page"]] * len(mots_page))

    for debut in range(0, len(mots_document), pas):

        fin = min(debut + taille_chunk, len(mots_document))

        texte_chunk = " ".join(mots_document[debut:fin])

        if not texte_chunk.strip():
            continue

        chunks.append({
            "texte": texte_chunk,
            "source": nom_document,
            "page_debut": pages_mots[debut],
            "page_fin": pages_mots[fin - 1],
            "nombre_mots": fin - debut
        })

        if fin == len(mots_document):
            break

print("Taille d'un chunk :", taille_chunk, "mots")
print("Chevauchement :", overlap, "mots")
print("Pas de déplacement :", pas, "mots")
print("\nNombre total de chunks :", len(chunks))

Taille d'un chunk : 400 mots
Chevauchement : 50 mots
Pas de déplacement : 350 mots

Nombre total de chunks : 14


### 3.3 -Afficher un exemple de chunk

In [14]:
print("EXEMPLE DE CHUNK")
print("=" * 60)

exemple = chunks[0]

print("Source :", exemple["source"])
print("Page de début :", exemple["page_debut"])
print("Page de fin :", exemple["page_fin"])
print("Nombre de mots :", exemple["nombre_mots"])

print("\nContenu :")
print(exemple["texte"])

EXEMPLE DE CHUNK
Source : 01_guide_pfe.pdf
Page de début : 1
Page de fin : 1
Nombre de mots : 400

Contenu :
ISIT – Guide PFE – 2026-2027 – v3.0 Page 1 Guide du Projet de Fin d'Études (PFE) – ISIT – 2026-2027 Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01 1. Définition et objectifs du PFE Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ingénieur de l'ISIT, réalisée en troisième année. Il vaut 30 crédits ECTS et se déroule en entreprise ou en laboratoire de recherche. Son objectif est de mettre l'étudiant en situation professionnelle : analyser un besoin, concevoir, développer, tester et documenter une solution complète. Le PFE est obligatoire pour obtenir le diplôme d'ingénieur. 2. Conditions d'inscription au PFE Pour s'inscrire au PFE, l'étudiant doit avoir validé tous les modules des semestres 1 à 5 du cycle ingénieur, ou avoir au maximum un module non validé avec accord écrit du chef de département. Il doit aussi avoir validé le 

In [15]:
print("Type :", type(chunks[0]["texte"]))
print("Longueur du texte :", len(chunks[0]["texte"]))
print("Nombre de caractères :", len(chunks[0]["texte"].strip()))

print("\nDébut du chunk :")
print(repr(chunks[0]["texte"][:500]))

print("\nContenu complet :")
print(chunks[0]["texte"])

Type : <class 'str'>
Longueur du texte : 2442
Nombre de caractères : 2442

Début du chunk :
"ISIT – Guide PFE – 2026-2027 – v3.0 Page 1 Guide du Projet de Fin d'Études (PFE) – ISIT – 2026-2027 Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01 1. Définition et objectifs du PFE Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ingénieur de l'ISIT, réalisée en troisième année. Il vaut 30 crédits ECTS et se déroule en entreprise ou en laboratoire de recherche. Son objectif est de mettre l'étudiant en situation professionnelle : analyser un be"

Contenu complet :
ISIT – Guide PFE – 2026-2027 – v3.0 Page 1 Guide du Projet de Fin d'Études (PFE) – ISIT – 2026-2027 Institut Supérieur d'Informatique et de Technologies (ISIT) · Version 3.0 · 2026-09-01 1. Définition et objectifs du PFE Le Projet de Fin d'Études (PFE) est l'épreuve finale du cycle ingénieur de l'ISIT, réalisée en troisième année. Il vaut 30 crédits ECTS et se déroule en entreprise o

## Étape 4 — Construction du système de recherche

### 4.1 Génération des embeddings

Chaque chunk est transformé en un vecteur numérique à l'aide d'un modèle Sentence Transformers multilingue. Ces vecteurs permettent de comparer le sens des textes et de retrouver les passages pertinents pour une question.

In [16]:

import numpy as np
from sentence_transformers import SentenceTransformer

# Vérification des chunks
if not chunks:
    raise ValueError("Aucun chunk disponible. Vérifiez l'étape de découpage.")

if not chunks[0]["texte"].strip():
    raise ValueError("Le premier chunk est vide. Vérifiez l'extraction des PDF.")

# Chargement du modèle multilingue
MODELE_EMBEDDING = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(MODELE_EMBEDDING)

# Extraction du texte de chaque chunk
textes_chunks = [chunk["texte"] for chunk in chunks]

# Transformation des chunks en vecteurs numériques
embeddings = embedding_model.encode(
    textes_chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Nombre de chunks :", len(chunks))
print("Forme de la matrice des embeddings :", embeddings.shape)
print("Dimensions d'un embedding :", embeddings.shape[1])
print("Embeddings générés avec succès.")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\rymgh\PFE_Navigator\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rymgh\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Nombre de chunks : 14
Forme de la matrice des embeddings : (14, 384)
Dimensions d'un embedding : 384
Embeddings générés avec succès.


### 4.2 Recherche des passages pertinents

Lorsqu'une question est posée, elle est transformée en embedding. Nous calculons ensuite sa similarité cosinus avec les embeddings des chunks.

Les passages sont classés par ordre de similarité décroissante. Nous récupérons les trois passages les plus pertinents (Top-k avec k = 3).

In [17]:

def rechercher(question, k=3):
    # Transformer la question en embedding
    embedding_question = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculer la similarité cosinus
    scores = np.dot(embeddings, embedding_question[0])

    # Récupérer les indices des k meilleurs chunks
    meilleurs_indices = np.argsort(scores)[::-1][:k]

    resultats = []

    for indice in meilleurs_indices:
        resultats.append({
            "score": float(scores[indice]),
            "texte": chunks[indice]["texte"],
            "source": chunks[indice]["source"],
            "page_debut": chunks[indice]["page_debut"],
            "page_fin": chunks[indice]["page_fin"]
        })

    return resultats


### 4.3 Évaluation de la recherche

Nous testons la recherche sur une question dont la réponse figure dans notre corpus documentaire. Nous affichons les trois passages retrouvés, leurs scores de similarité et leurs sources.

In [18]:

question_test = "Quelle est la durée du Projet de Fin d'Études ?"

resultats = rechercher(question_test, k=3)

print("QUESTION :", question_test)
print("=" * 70)

for i, resultat in enumerate(resultats, start=1):
    print(f"\nPASSAGE {i}")
    print("-" * 50)
    print("Score de similarité :", round(resultat["score"], 4))
    print("Document :", resultat["source"])
    print("Pages :", resultat["page_debut"], "-", resultat["page_fin"])
    print("Contenu :", resultat["texte"][:700])


QUESTION : Quelle est la durée du Projet de Fin d'Études ?

PASSAGE 1
--------------------------------------------------
Score de similarité : 0.7104
Document : 01_guide_pfe.pdf
Pages : 1 - 2
Contenu : validée par la commission des PFE avant le 4 décembre 2026. Une fiche incomplète est renvoyée à l'étudiant avec des remarques ; il dispose de 7 jours pour la corriger. 6. Encadrement : rôles et responsabilités Chaque étudiant a deux encadrants. • L'encadrant académique est un enseignant de l'ISIT, affecté au plus tard le 16 novembre 2026. Il valide la démarche scientifique, relit le rapport et organise au moins 4 réunions de suivi. • L'encadrant professionnel est un ingénieur ou un cadre de la structure d'accueil, ayant au moins 3 ans d'expérience. Il définit les tâches au quotidien et évalue l'étudiant. ISIT – Guide PFE – 2026-2027 – v3.0 Page 2 L'étudiant doit envoyer à son encadrant acadé

PASSAGE 2
--------------------------------------------------
Score de similarité : 0.6587
Docume

premier problème de pertinence.

1. Analyse des résultats


1	: 0.7104 : Bonne source, mais l'extrait affiché ne parle pas de la durée

2	0.6587	: Extrait sur l'organisation de l'année

3	0.6574	:	Extrait sur la soutenance

Le passage 1 provient bien du guide PFE, mais attention : un score élevé ne garantit pas que le passage contient la réponse.

Nous ne devons donc pas encore considérer la recherche comme validée.

Diagnostiquer le problème

Cette cellule vérifie si les passages retrouvés contiennent le mot « semaine » et affiche le contexte lorsqu'il est présent

In [19]:

for i, resultat in enumerate(resultats, start=1):
    texte = resultat["texte"].lower()

    print(f"\nPASSAGE {i}")
    print("Source :", resultat["source"])
    print("Contient le mot 'semaine' :", "semaine" in texte)

    if "semaine" in texte:
        position = texte.find("semaine")
        debut = max(0, position - 200)
        fin = min(len(texte), position + 300)
        print("Contexte :", resultat["texte"][debut:fin])



PASSAGE 1
Source : 01_guide_pfe.pdf
Contient le mot 'semaine' : True
Contexte : nce. Il définit les tâches au quotidien et évalue l'étudiant. ISIT – Guide PFE – 2026-2027 – v3.0 Page 2 L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux semaines. 7. Documents à fournir pendant le PFE • La convention de PFE, signée par les trois parties (étudiant, entreprise, ISIT) avant le démarrage. • Un rapport de mi-parcours de 5 à 10 pages, à rendre avant le 15 avril 2027. • Le pré-rapport complet, à rendre avant le 24 mai 2027 pour relecture p

PASSAGE 2
Source : 05_calendrier_universitaire.pdf
Contient le mot 'semaine' : False

PASSAGE 3
Source : 04_guide_soutenance.pdf
Contient le mot 'semaine' : False


In [20]:

# Rechercher les phrases contenant des mots liés à la durée
mots_cles = ["semaine", "semaines", "durée", "duree"]

print("RECHERCHE DANS LE GUIDE PFE")
print("=" * 70)

for page in pages_data:
    if page["source"] == "01_guide_pfe.pdf":
        phrases = re.split(r'(?<=[.!?])\s+', page["texte"])

        for phrase in phrases:
            if any(mot in phrase.lower() for mot in mots_cles):
                print(f"\nPage {page['page']} :")
                print(phrase)


RECHERCHE DANS LE GUIDE PFE

Page 1 :
Durée et période du PFE Le PFE dure au minimum 20 semaines et au maximum 24 semaines, soit environ 5 à 6 mois.

Page 1 :
Un temps plein de 35 à 40 heures par semaine est attendu.

Page 2 :
ISIT – Guide PFE – 2026-2027 – v3.0 Page 2 L'étudiant doit envoyer à son encadrant académique un point d'avancement écrit toutes les deux semaines.


In [21]:

print("RECHERCHE DE L'INFORMATION EXACTE DANS LES CHUNKS")
print("=" * 70)

for i, chunk in enumerate(chunks):
    texte = chunk["texte"].lower()

    if "20 semaines" in texte or "24 semaines" in texte:
        print("\nChunk numéro :", i)
        print("Source :", chunk["source"])
        print("Pages :", chunk["page_debut"], "-", chunk["page_fin"])
        print("Contenu :")

        position = texte.find("20 semaines")
        if position == -1:
            position = texte.find("24 semaines")

        debut = max(0, position - 200)
        fin = min(len(texte), position + 300)

        print(chunk["texte"][debut:fin])


RECHERCHE DE L'INFORMATION EXACTE DANS LES CHUNKS

Chunk numéro : 0
Source : 01_guide_pfe.pdf
Pages : 1 - 1
Contenu :
 avec accord écrit du chef de département. Il doit aussi avoir validé le stage technique de deuxième année et être à jour de ses frais d'inscription. 3. Durée et période du PFE Le PFE dure au minimum 20 semaines et au maximum 24 semaines, soit environ 5 à 6 mois. Pour l'année 2026-2027, il commence le 1er février 2027 et doit se terminer au plus tard le 11 juin 2027, date limite de dépôt du rapport final. Un temps plein de 35 à 40 heures par semaine est attendu. 4. Choix du sujet et de la struct

Chunk numéro : 3
Source : 02_reglement_stages.pdf
Pages : 1 - 1
Contenu :
um, en juillet ou août, pour découvrir le monde de l'entreprise. • Stage technique (2e année) : 6 semaines minimum, en juillet ou août, avec une mission technique encadrée. • PFE (3e année) : de 20 à 24 semaines, de février à juin (voir le Guide PFE). Un stage ne peut pas dépasser 6 mois dans la même stru

In [22]:

question_corrigee = (
    "Combien de semaines dure le Projet de Fin d'Études ? "
    "Quelle est sa durée minimale et maximale ?"
)

resultats_corriges = rechercher(question_corrigee, k=3)

print("QUESTION INITIALE :", question_test)
print("QUESTION AMELIOREE :", question_corrigee)
print("=" * 70)

for i, resultat in enumerate(resultats_corriges, start=1):
    print(f"\nPASSAGE {i}")
    print("-" * 50)
    print("Score :", round(resultat["score"], 4))
    print("Document :", resultat["source"])
    print("Pages :", resultat["page_debut"], "-", resultat["page_fin"])
    print("Contient '20 semaines' :", "20 semaines" in resultat["texte"].lower())
    print("Contient '24 semaines' :", "24 semaines" in resultat["texte"].lower())
    print("Contenu :", resultat["texte"][:500])


QUESTION INITIALE : Quelle est la durée du Projet de Fin d'Études ?
QUESTION AMELIOREE : Combien de semaines dure le Projet de Fin d'Études ? Quelle est sa durée minimale et maximale ?

PASSAGE 1
--------------------------------------------------
Score : 0.6815
Document : 01_guide_pfe.pdf
Pages : 1 - 2
Contient '20 semaines' : False
Contient '24 semaines' : False
Contenu : validée par la commission des PFE avant le 4 décembre 2026. Une fiche incomplète est renvoyée à l'étudiant avec des remarques ; il dispose de 7 jours pour la corriger. 6. Encadrement : rôles et responsabilités Chaque étudiant a deux encadrants. • L'encadrant académique est un enseignant de l'ISIT, affecté au plus tard le 16 novembre 2026. Il valide la démarche scientifique, relit le rapport et organise au moins 4 réunions de suivi. • L'encadrant professionnel est un ingénieur ou un cadre de la s

PASSAGE 2
--------------------------------------------------
Score : 0.6785
Document : 04_guide_soutenance.pdf
Pages : 2 -

In [23]:

# Transformer la question initiale en embedding
embedding_question = embedding_model.encode(
    [question_test],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Calculer les scores de tous les chunks
scores = np.dot(embeddings, embedding_question[0])

# Trier les chunks du plus pertinent au moins pertinent
classement = np.argsort(scores)[::-1]

# Afficher les 10 premiers résultats
print("CLASSEMENT DES CHUNKS")
print("=" * 70)

for rang, indice in enumerate(classement[:10], start=1):
    chunk = chunks[indice]

    print(f"\nRang : {rang}")
    print("Numéro du chunk :", indice)
    print("Score :", round(float(scores[indice]), 4))
    print("Source :", chunk["source"])
    print("Pages :", chunk["page_debut"], "-", chunk["page_fin"])
    print("Contient 20 semaines :", "20 semaines" in chunk["texte"].lower())
    print("Contient 24 semaines :", "24 semaines" in chunk["texte"].lower())


CLASSEMENT DES CHUNKS

Rang : 1
Numéro du chunk : 1
Score : 0.7104
Source : 01_guide_pfe.pdf
Pages : 1 - 2
Contient 20 semaines : False
Contient 24 semaines : False

Rang : 2
Numéro du chunk : 12
Score : 0.6587
Source : 05_calendrier_universitaire.pdf
Pages : 1 - 2
Contient 20 semaines : False
Contient 24 semaines : False

Rang : 3
Numéro du chunk : 10
Score : 0.6574
Source : 04_guide_soutenance.pdf
Pages : 2 - 3
Contient 20 semaines : False
Contient 24 semaines : False

Rang : 4
Numéro du chunk : 0
Score : 0.6224
Source : 01_guide_pfe.pdf
Pages : 1 - 1
Contient 20 semaines : True
Contient 24 semaines : True

Rang : 5
Numéro du chunk : 13
Score : 0.62
Source : 05_calendrier_universitaire.pdf
Pages : 2 - 2
Contient 20 semaines : False
Contient 24 semaines : False

Rang : 6
Numéro du chunk : 5
Score : 0.6074
Source : 02_reglement_stages.pdf
Pages : 2 - 3
Contient 20 semaines : False
Contient 24 semaines : False

Rang : 7
Numéro du chunk : 3
Score : 0.5789
Source : 02_reglement_stages.pdf

Le modèle peut donc attribuer un score élevé à un passage globalement lié au PFE, même s'il ne contient pas la réponse recherchée.

2. Correction proposée : réduire la taille des chunks

Nous allons tester des chunks plus petits, de 200 mots avec un chevauchement de 40 mots.

Pourquoi ?
- Les passages seront plus ciblés.
- Chaque chunk contiendra moins de sujets différents.
- Le modèle pourra mieux distinguer le passage qui parle de la durée du PFE.
Nous allons garder le même modèle d'embedding pour comparer correctement les deux méthodes.

In [24]:

# Nouvelle stratégie de découpage
nouvelle_taille_chunk = 200
nouvel_overlap = 40
nouveau_pas = nouvelle_taille_chunk - nouvel_overlap

nouveaux_chunks = []

for nom_document, pages in documents.items():
    mots_document = []
    pages_mots = []

    for page in pages:
        mots_page = page["texte"].split()
        mots_document.extend(mots_page)
        pages_mots.extend([page["page"]] * len(mots_page))

    for debut in range(0, len(mots_document), nouveau_pas):
        fin = min(debut + nouvelle_taille_chunk, len(mots_document))

        texte_chunk = " ".join(mots_document[debut:fin])

        if texte_chunk.strip():
            nouveaux_chunks.append({
                "texte": texte_chunk,
                "source": nom_document,
                "page_debut": pages_mots[debut],
                "page_fin": pages_mots[fin - 1],
                "nombre_mots": fin - debut
            })

        if fin == len(mots_document):
            break

print("Ancien nombre de chunks :", len(chunks))
print("Nouveau nombre de chunks :", len(nouveaux_chunks))
print("Nouvelle taille :", nouvelle_taille_chunk)
print("Nouvel overlap :", nouvel_overlap)



Ancien nombre de chunks : 14
Nouveau nombre de chunks : 26
Nouvelle taille : 200
Nouvel overlap : 40


In [25]:

nouveaux_textes = [chunk["texte"] for chunk in nouveaux_chunks]

nouveaux_embeddings = embedding_model.encode(
    nouveaux_textes,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Nombre de nouveaux chunks :", len(nouveaux_chunks))
print("Forme des nouveaux embeddings :", nouveaux_embeddings.shape)
print("Nouvelle représentation vectorielle générée.")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Nombre de nouveaux chunks : 26
Forme des nouveaux embeddings : (26, 384)
Nouvelle représentation vectorielle générée.


In [26]:

def rechercher_corrigee(question, k=3):
    embedding_question = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = np.dot(nouveaux_embeddings, embedding_question[0])
    meilleurs_indices = np.argsort(scores)[::-1][:k]

    resultats = []

    for indice in meilleurs_indices:
        resultats.append({
            "score": float(scores[indice]),
            "texte": nouveaux_chunks[indice]["texte"],
            "source": nouveaux_chunks[indice]["source"],
            "page_debut": nouveaux_chunks[indice]["page_debut"],
            "page_fin": nouveaux_chunks[indice]["page_fin"]
        })

    return resultats


resultats_apres = rechercher_corrigee(question_test, k=3)

print("RECHERCHE APRÈS CORRECTION")
print("=" * 70)

for i, resultat in enumerate(resultats_apres, start=1):
    texte = resultat["texte"].lower()

    print(f"\nPASSAGE {i}")
    print("Score :", round(resultat["score"], 4))
    print("Document :", resultat["source"])
    print("Pages :", resultat["page_debut"], "-", resultat["page_fin"])
    print("Contient 20 semaines :", "20 semaines" in texte)
    print("Contient 24 semaines :", "24 semaines" in texte)
    print("Contenu :", resultat["texte"][:500])


RECHERCHE APRÈS CORRECTION

PASSAGE 1
Score : 0.7195
Document : 01_guide_pfe.pdf
Pages : 1 - 1
Contient 20 semaines : True
Contient 24 semaines : True
Contenu : 3. Durée et période du PFE Le PFE dure au minimum 20 semaines et au maximum 24 semaines, soit environ 5 à 6 mois. Pour l'année 2026-2027, il commence le 1er février 2027 et doit se terminer au plus tard le 11 juin 2027, date limite de dépôt du rapport final. Un temps plein de 35 à 40 heures par semaine est attendu. 4. Choix du sujet et de la structure d'accueil L'étudiant trouve lui-même sa structure d'accueil (entreprise, start-up, administration ou laboratoire) ou choisit un sujet dans la banq

PASSAGE 2
Score : 0.6821
Document : 01_guide_pfe.pdf
Pages : 1 - 2
Contient 20 semaines : False
Contient 24 semaines : False
Contenu : titre, la problématique, les objectifs, les technologies envisagées, le planning prévisionnel et les coordonnées de l'encadrant professionnel. La fiche est signée par l'entreprise puis soumise au départ

Attention : 
la réduction de la taille des chunks est une hypothèse de correction, pas une garantie. Nous devons vérifier les résultats réels avant de conclure que le problème est résolu

In [27]:
question_test = "Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?"

resultats_apres = rechercher_corrigee(question_test, k=3)

print("QUESTION :", question_test)
print("=" * 70)

for i, resultat in enumerate(resultats_apres, start=1):
    print(f"\nPASSAGE {i}")
    print("Score :", round(resultat["score"], 4))
    print("Document :", resultat["source"])
    print("Pages :", resultat["page_debut"], "-", resultat["page_fin"])
    print("Contenu :", resultat["texte"][:700])

QUESTION : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

PASSAGE 1
Score : 0.8276
Document : 05_calendrier_universitaire.pdf
Pages : 1 - 2
Contenu : des encadrants académiques : 16 novembre 2026. • Date limite de validation de la fiche PFE : 4 décembre 2026. ISIT – Calendrier universitaire – 2026-2027 – v1.4 Page 2 • Début du PFE : 1er février 2027. • Date limite de changement de sujet : 1er mars 2027. • Rapport de mi-parcours : 15 avril 2027. • Dépôt du pré-rapport : 24 mai 2027. • Dépôt du rapport final : 11 juin 2027. • Publication du planning des soutenances : 14 juin 2027. • Soutenances (session principale) : du 21 juin au 2 juillet 2027. • Dépôt de la version corrigée : 15 jours après la soutenance. • Soutenances (session de rattrapage) : du 13 au 17 septembre 2027. 6. Calendrier des stages d'été • Dépôt des demandes de convention 

PASSAGE 2
Score : 0.7931
Document : 01_guide_pfe.pdf
Pages : 2 - 2
Contenu : pendant le PFE

##Analyse des résultats après correction

Le passage 2 contient exactement l'information recherchée :
- Date limite : 11 juin 2027.
- Remise numérique : sur la plateforme ISIT-Stages, au format PDF.
- Remise physique : un exemplaire relié au secrétariat du département.

In [28]:

question_comparaison = questions[0]

resultats_avant = rechercher(question_comparaison, k=3)
resultats_apres = rechercher_corrigee(question_comparaison, k=3)

print("QUESTION :", question_comparaison)
print("\n========== AVANT CORRECTION ==========")

for i, r in enumerate(resultats_avant, start=1):
    print(f"\nPassage {i} | Score : {r['score']:.4f}")
    print("Source :", r["source"])
    print("Contient la date :", "11 juin 2027" in r["texte"].lower())
    print("Contient PDF :", "pdf" in r["texte"].lower())
    print("Contient relié :", "relié" in r["texte"].lower())

print("\n========== APRÈS CORRECTION ==========")

for i, r in enumerate(resultats_apres, start=1):
    print(f"\nPassage {i} | Score : {r['score']:.4f}")
    print("Source :", r["source"])
    print("Contient la date :", "11 juin 2027" in r["texte"].lower())
    print("Contient PDF :", "pdf" in r["texte"].lower())
    print("Contient relié :", "relié" in r["texte"].lower())


QUESTION : Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

========== AVANT CORRECTION ==========

Passage 1 | Score : 0.7928
Source : 05_calendrier_universitaire.pdf
Contient la date : True
Contient PDF : False
Contient relié : False

Passage 2 | Score : 0.6616
Source : 01_guide_pfe.pdf
Contient la date : False
Contient PDF : False
Contient relié : False

Passage 3 | Score : 0.6053
Source : 01_guide_pfe.pdf
Contient la date : True
Contient PDF : False
Contient relié : False

========== APRÈS CORRECTION ==========

Passage 1 | Score : 0.8276
Source : 05_calendrier_universitaire.pdf
Contient la date : True
Contient PDF : False
Contient relié : False

Passage 2 | Score : 0.7931
Source : 01_guide_pfe.pdf
Contient la date : True
Contient PDF : True
Contient relié : True

Passage 3 | Score : 0.7889
Source : 01_guide_pfe.pdf
Contient la date : False
Contient PDF : False
Contient relié : False


## étape 5 - Ajouter la génération

### 5.1 Construction du prompt RAG

Les passages récupérés par le système de recherche sont transmis au LLM avec la question de l'étudiant.

Le prompt impose au modèle de :
- Répondre uniquement à partir des documents fournis.
- Ne pas inventer d'informations.
- Indiquer les sources utilisées.
- Répondre « Je ne sais pas » si les documents ne contiennent pas la réponse.

In [29]:

def construire_prompt(question, resultats):
    contexte = ""

    for i, resultat in enumerate(resultats, start=1):
        contexte += (
            f"\n[Passage {i}]\n"
            f"Document : {resultat['source']}\n"
            f"Pages : {resultat['page_debut']} - {resultat['page_fin']}\n"
            f"Contenu : {resultat['texte']}\n"
        )

    prompt = f"""
Tu es PFE Navigator, un assistant spécialisé dans les
informations relatives au Projet de Fin d'Études de l'ISIT.

Ta mission est de répondre aux questions des étudiants
en utilisant exclusivement les passages documentaires fournis.

RÈGLES :
1. Utilise uniquement les informations présentes dans le contexte.
2. N'invente aucune information.
3. Si la réponse n'est pas présente dans les passages,
   réponds exactement : "Je ne sais pas d'après les documents fournis."
4. Cite les documents et les pages utilisés.
5. Réponds clairement et de manière structurée.

CONTEXTE DOCUMENTAIRE :
{contexte}

QUESTION DE L'ÉTUDIANT :
{question}

RÉPONSE :
"""
    return prompt


### 5.2 Génération de la réponse avec Gemini

Le prompt construit à partir des passages pertinents est envoyé au modèle Gemini. Celui-ci génère une réponse en respectant les instructions et en citant les sources documentaires.

In [31]:

def repondre_avec_rag(question, k=3):
    # Étape 1 : rechercher les passages pertinents
    resultats = rechercher_corrigee(question, k=k)

    # Étape 2 : construire le prompt
    prompt = construire_prompt(question, resultats)

    # Étape 3 : envoyer le prompt à Gemini
    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return {
        "question": question,
        "reponse": response.text,
        "sources": [
            {
                "document": r["source"],
                "pages": f"{r['page_debut']} - {r['page_fin']}"
            }
            for r in resultats
        ]
    }


### Premier test RAG avec ta question

Nous allons reprendre ta question numéro 1 :
Quelle est la date limite de dépôt du rapport final de PFE, et sous quelle forme faut-il le remettre ?

In [32]:

question = questions[0]

resultat_rag = repondre_avec_rag(question, k=3)

print("QUESTION :", resultat_rag["question"])
print("\nRÉPONSE DE PFE NAVIGATOR :")
print(resultat_rag["reponse"])

print("\nSOURCES UTILISÉES :")
for source in resultat_rag["sources"]:
    print("-", source["document"], "| Pages :", source["pages"])


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 11h11m20.557242875s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per day limit exceeded on Free Tier. See https://aistudio.